# Example 10.4.1 / 1: Convex epigraphs

Companion notebook for **10.4.1 Convexity and Piecewise-Linear Reformulations** in the AMPL lab Documentation, section "Convex epigraphs". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Convex epigraphs

For a convex function $f(x)=\max_r\{a_rx+b_r\}$, the epigraph formulation is

$$y\ge a_rx+b_r\quad\text{for every segment }r.$$

Minimizing `y` makes at least one inequality tight, so the formulation is exact without binary variables.

The equality is exact only if the rest of the model has no incentive or requirement to keep `y`
above the graph. More generally, replacing `y = f(x)` by an epigraph is valid when every feasible
use of `y` is monotone in the direction that pushes it down. Check every objective and constraint
containing `y`, not just the primary objective coefficient.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param demand >= 0;
var Quantity >= demand;
var Cost;

subject to RegularRate: Cost >= 20 * Quantity;
subject to OvertimeRate: Cost >= 35 * Quantity - 600;

minimize TotalCost: Cost;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["demand"] = 48

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.var["Cost"].value())

HiGHS 1.14.0: optimal solution; objective 1080
0 simplex iterations
0 barrier iterations
solved
1080.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL:

In [6]:
%%writefile data.dat
param demand := 48;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Quantity, Cost;

HiGHS 1.14.0: optimal solution; objective 1080
0 simplex iterations
0 barrier iterations
Quantity = 48
Cost = 1080

